# SFERE — interactive run (JAX)

The SFERE model (van de Vijsel et al. 2023) in an interactive notebook: run the
cells top to bottom to watch the channel network emerge live, then animate the
frames and save the outputs. This is the same physics as the published OpenCL
notebook at the repo root, rebuilt on the JAX package in `sfere/`.

Requires the package installed (`pip install -e .`, which also provides the
kernel dependencies) and the `sfere-jax` kernel selected.

The config below is a **notebook-scale demo** (~1 min on one GPU). The commented alternatives show the RUN9 defaults of the paper.

In [1]:
import numpy as np

from sfere.config import GradientOptions, GridConfig, PhysParams, RunConfig, SimConfig
from sfere.live import LiveFigure, html_animation
from sfere.model import Params
from sfere.simulate import Simulation
from sfere.state import init_conditions

# Notebook-scale demo (~1 min on one GPU). For a RUN9-style run use:
#   grid = GridConfig()                          # 2048 x 1024
#   sim  = SimConfig()                           # EndTime=70580, NumFrames=100
grid = GridConfig(Block_Size_X=16, Block_Size_Y=16, Block_Number_X=32, Block_Number_Y=16)  # 512 x 256
phys = PhysParams()
gradients = GradientOptions()
sim = SimConfig(EndTime=5000.0, NumFrames=20, dT=0.0125)
run = RunConfig(RUN=9, SUBRUNa=1, SUBRUNb=1, outdir="RUN9")
params = Params(phys=phys, gradients=gradients, dX=grid.dX, dY=grid.dY, dT=sim.dT)
print(f"{grid.Grid_Width} x {grid.Grid_Height}, {sim.NumFrames} frames, {sim.steps_per_frame} steps/frame")

ModuleNotFoundError: No module named 'sfere'

In [ ]:
state = init_conditions(phys, grid, vegetation=True, seed=2)
simulation = Simulation(state, params, sim)

## Watch it evolve

The 8-panel figure is created once and updated in place after every frame,
with the elapsed time in years in the suptitle — like the original notebook's
animation, but live as the simulation runs.

In [ ]:
live = LiveFigure(simulation, run)
live.show()                      # the initial condition

In [ ]:
for _ in range(sim.NumFrames):
    simulation.advance()         # each frame = sim.steps_per_frame model steps
    live.show()

Re-executing the loop cell **continues** the same run: `Simulation` holds
the state on the device, so you can pause, look around, and run more frames
(`tides_elapsed` tracks the simulated time even beyond `sim.EndTime`).

In [ ]:
html_animation(simulation, run)  # scrub through every frame captured so far

## Save

The original notebook's filenames, in `run.outdir` (the directory is created).
Pass `npy=False` etc. to skip parts.

In [ ]:
print(simulation.save(run))

### Notes

- Grid dimension must be powers of 2, >= 16 (per-axis `Block_Size * Block_Number`).
- Frames live on the host as float32: ~6.7 GB for the RUN9 default (100 frames,
  2048x1024, 8 variables) — same bookkeeping as the original.
- Coordinate convention: as in the original code, X runs left->right and Y
  top->bottom (the manuscript axes are swapped); see README "Notes for newcomers".